# Qwen3-8B-AWQ + vLLM trên Colab T4 → API cho DACNTT

**MODEL-SERVE-01 · REVIEW · STATIC VALIDATED — NOT GPU TESTED**

Chọn **Runtime → Change runtime type → T4 GPU**, **Run all** và cấp quyền Google Drive.
Notebook tải/dùng cache **Qwen/Qwen3-8B-AWQ (4-bit)**, start OpenAI-compatible API,
thử HTTPS và lưu **`MyDrive/DACNTT/api_info.json`**. API key không hiển thị;
không chia sẻ/commit file này. **Mục tiêu 24K chưa được kiểm chứng trên T4.**

Giữ vLLM **0.10.2**, Python riêng **3.11**, AWQ/float16 activations, engine V0/XFormers,
eager mode, concurrency **1**, non-thinking. Context mặc định **24576 tokens** gồm
input và output; có thể chọn 16K/8K thủ công khi thiếu VRAM/KV-cache.
Không build source, không tải checkpoint FP16, không tự retry/đổi context.
Chunked prefill và prefix caching tắt theo lỗi kernel shared-memory đã gặp ở phiên T4 trước.

Cache **8B riêng**, không dùng hoặc xóa cache14B. Lần đầu chuyển model cần tải weights8B.
Giữ GPU memory utilization **0.90**; chưa bật CPU offload hoặc FP8 KV-cache.

Cloudflare Quick Tunnel chỉ dùng thử khi chính sách môi trường cho phép; không SLA,
URL đổi mỗi lần start, không SSE: client **stream=false**. Không giữ phiên sống/vượt chặn.
Nếu HTTPS không xác minh được, ghi `LOCAL_ONLY`, `BASE_URL: null`.

[Model](https://huggingface.co/Qwen/Qwen3-8B-AWQ) ·
[vLLM0.10.2](https://docs.vllm.ai/en/v0.10.2/getting_started/installation/gpu.html) ·
[AWQ SM75](https://github.com/vllm-project/vllm/blob/v0.10.2/vllm/model_executor/layers/quantization/awq.py) ·
[Quick Tunnel](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) ·
[Colab](https://research.google.com/colaboratory/faq.html)


## 1. Chuẩn bị GPU và Google Drive

Mặc định Qwen3-8B-AWQ với context **24K = 24576 tokens**. Khi chuyển từ14B,
chạy lại từ cell này → dùng lại môi trường vLLM → chuẩn bị model8B → Start API → Test API.
Nếu thiếu VRAM/KV-cache, chọn16K hoặc8K rồi start lại; không tự giảm context.
HTTPS có thể tắt để chỉ test local.


In [ ]:
import hashlib
import json
import os
import re
import secrets
import shutil
import signal
import socket
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive
from IPython.display import display

# fmt: off
#@title Chuẩn bị
CONTEXT_SIZE = 24576 #@param [8192, 16384, 24576] {type:"raw"}
ENABLE_HTTPS = True #@param {type:"boolean"}
# fmt: on

ROOT = Path("/content/dacntt_qwen")
ROOT.mkdir(exist_ok=True)
MODEL_NAME = "qwen3-8b-awq-t4"
MODEL_REPO = "Qwen/Qwen3-8B-AWQ"
MODEL_REVISION = "4da05a8edb55c6046cce958586c33b61da07bb79"
PORT = 8000
API = f"http://127.0.0.1:{PORT}"
SERVER_READY = False
BASE_URL = None


def process_identity(pid):
    try:
        return Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[19]
    except FileNotFoundError:
        return None


def stop_process(kind):
    state_file = ROOT / f"{kind}.json"
    if not state_file.exists():
        return
    state = json.loads(state_file.read_text())
    pid = state["pid"]
    if state["starttime"] and process_identity(pid) == state["starttime"]:
        args = Path(f"/proc/{pid}/cmdline").read_bytes().split(b"\0")
        if str(state["binary"]).encode() not in args or os.getpgid(pid) != pid:
            raise RuntimeError("Không xác minh được process sở hữu. Hãy Restart Session.")
        os.killpg(pid, signal.SIGTERM)
        proc = globals().get(kind.upper() + "_PROCESS")
        for _ in range(30):
            if proc is not None and proc.pid == pid:
                proc.poll()
            if process_identity(pid) is None:
                break
            time.sleep(0.1)
        else:
            if process_identity(pid) == state["starttime"]:
                os.killpg(pid, signal.SIGKILL)
        if proc is not None and proc.pid == pid:
            proc.wait(timeout=10)
    state_file.unlink(missing_ok=True)


def save_api_info(status, base_url=None):
    info = {
        "BASE_URL": base_url,
        "API_KEY": API_KEY,
        "MODEL_NAME": MODEL_NAME,
        "STATUS": status,
        "LOCAL_BASE_URL": API + "/v1",
        "UPDATED_AT": datetime.now(UTC).isoformat(),
    }
    temporary = INFO_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(info, indent=2), encoding="utf-8")
    os.replace(temporary, INFO_PATH)


def stop_api():
    global SERVER_READY, BASE_URL
    SERVER_READY, BASE_URL = False, None
    stop_process("tunnel")
    stop_process("server")
    if "INFO_PATH" in globals() and "API_KEY" in globals():
        save_api_info("STOPPED")


stop_api()
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/DACNTT")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
INFO_PATH = DRIVE_ROOT / "api_info.json"
API_KEY = secrets.token_urlsafe(32)
HEADERS = {"Authorization": "Bearer " + API_KEY}
# Xóa URL cũ khỏi bản ghi trước khi chuẩn bị phiên mới; không in API key.
save_api_info("PREPARING")


def gpu_memory():
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free,compute_cap",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=15,
    )
    fields = [v.strip() for v in result.stdout.splitlines()[0].split(",")]
    return dict(
        zip(["name", "total_MiB", "used_MiB", "free_MiB", "compute_cap"], fields, strict=True)
    )


try:
    GPU = gpu_memory()
except (OSError, subprocess.SubprocessError):
    raise RuntimeError("Không có NVIDIA GPU. Chọn T4 GPU rồi chạy lại.") from None
if float(GPU["compute_cap"]) < 7.5:
    raise RuntimeError("AWQ backend này yêu cầu NVIDIA compute capability >=7.5.")
print("GPU:", GPU)
if "T4" not in GPU["name"]:
    print("Cảnh báo: runtime được cấp GPU khác T4; kết quả áp dụng cho GPU trên.")
print("Context mục tiêu:", CONTEXT_SIZE, "| AWQ 4-bit | non-thinking")

## 2. Cài vLLM và chuẩn bị model

Cài wheels vào môi trường riêng; không đổi PyTorch của notebook, không cần restart kernel.
Lần đầu có thể mất vài phút và nhiều GiB disk cho thư viện. Nếu không có wheel phù hợp,
cell báo lỗi, không tự build source. Dùng lại môi trường đã cài trong cùng phiên.

Cache riêng **`MyDrive/DACNTT/models/Qwen3-8B-AWQ/`**, copy về `/content/models/`
trước khi load. Chưa có thì chỉ tải snapshot AWQ đã pin và lưu Drive để tái sử dụng.
Kiểm tra danh sách/kích thước files và config AWQ 4-bit; không hash toàn bộ weights.
Nếu cache hỏng nhưng vẫn đúng size, đổi tên folder cache rồi chạy lại để tải mới.


In [ ]:
# @title Cài vLLM (wheels, không build CUDA)
stop_api()
save_api_info("PREPARING")
VLLM_VERSION = "0.10.2"
ENV_DIR = ROOT / "vllm-env"
PYTHON = ENV_DIR / "bin/python"
INSTALL_LOG = ROOT / "install.log"
INSTALL_STAMP = ROOT / "vllm-install.json"
packages = [f"vllm=={VLLM_VERSION}", "transformers==4.55.2", "huggingface-hub==0.34.4"]


def install_command(command, timeout=1200):
    with INSTALL_LOG.open("a") as log:
        try:
            subprocess.run(
                list(map(str, command)),
                stdout=log,
                stderr=subprocess.STDOUT,
                check=True,
                timeout=timeout,
            )
        except (OSError, subprocess.SubprocessError) as exc:
            print(INSTALL_LOG.read_text(errors="replace")[-5000:].replace(API_KEY, "<REDACTED>"))
            raise RuntimeError(
                f"Cài vLLM thất bại ({type(exc).__name__}); xem {INSTALL_LOG}. Không build source."
            ) from None


if not (
    PYTHON.is_file()
    and INSTALL_STAMP.exists()
    and json.loads(INSTALL_STAMP.read_text()) == packages
):
    INSTALL_LOG.write_text("")
    install_command(
        [sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", "uv==0.9.5"], 180
    )
    if not PYTHON.is_file():
        install_command([sys.executable, "-m", "uv", "venv", "--python", "3.11", ENV_DIR], 300)
    install_command(
        [
            sys.executable,
            "-m",
            "uv",
            "pip",
            "install",
            "--python",
            PYTHON,
            "--only-binary=:all:",
            *packages,
        ]
    )
    INSTALL_STAMP.write_text(json.dumps(packages))
# Check CUDA trong subprocess; không import torch/vllm vào kernel notebook.
check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import torch, vllm; print('vLLM', vllm.__version__, 'torch', torch.__version__, 'CUDA', torch.version.cuda); "
        "assert torch.cuda.is_available(), 'CUDA unavailable'; "
        "print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))",
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print(check.stdout)
if check.returncode:
    print(check.stderr[-5000:])
    raise RuntimeError("Môi trường vLLM/CUDA chưa hợp lệ; xem lỗi trên. Chưa load weights.")

In [ ]:
# @title Chuẩn bị model AWQ — Drive hoặc Hugging Face
EXPECTED_FILES = {
    "config.json": 1042,
    "generation_config.json": 239,
    "merges.txt": 1671853,
    "model-00001-of-00002.safetensors": 4853922024,
    "model-00002-of-00002.safetensors": 1244659840,
    "model.safetensors.index.json": 74029,
    "tokenizer.json": 11422654,
    "tokenizer_config.json": 9732,
    "vocab.json": 2776833,
}
MODEL_DIR = Path("/content/models/Qwen3-8B-AWQ")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = DRIVE_ROOT / "models/Qwen3-8B-AWQ"
MODEL_SIZE = sum(EXPECTED_FILES.values())


def model_ok(directory):
    if not all(
        (directory / name).is_file() and (directory / name).stat().st_size == size
        for name, size in EXPECTED_FILES.items()
    ):
        return False
    try:
        config = json.loads((directory / "config.json").read_text())
        quant = config.get("quantization_config", {})
        return (
            config.get("model_type") == "qwen3"
            and quant.get("quant_method") == "awq"
            and quant.get("bits") == 4
        )
    except (OSError, ValueError):
        return False


def copy_model(source, destination):
    destination.mkdir(parents=True, exist_ok=True)
    missing = [
        (name, size)
        for name, size in EXPECTED_FILES.items()
        if not (destination / name).is_file() or (destination / name).stat().st_size != size
    ]
    if shutil.disk_usage(destination).free < sum(size for _, size in missing) + 512 * 2**20:
        raise OSError("Không đủ disk space để copy model.")
    for name, size in missing:
        temporary = (destination / name).with_suffix(".copying")
        try:
            shutil.copyfile(source / name, temporary)
            if temporary.stat().st_size != size:
                raise ValueError("Copy thiếu dữ liệu: " + name)
            os.replace(temporary, destination / name)
        finally:
            temporary.unlink(missing_ok=True)
    # Config nhỏ luôn copy lại, kể cả khi cache lỗi nhưng giữ nguyên kích thước.
    shutil.copyfile(source / "config.json", destination / "config.json")
    if not model_ok(destination):
        raise ValueError("Snapshot AWQ chưa hợp lệ sau copy.")


if not model_ok(MODEL_DIR):
    try:
        if model_ok(DRIVE_DIR):
            print("Dùng cache AWQ trên Drive → copy local.")
            copy_model(DRIVE_DIR, MODEL_DIR)
    except OSError as exc:
        print("Không copy được Drive:", type(exc).__name__, "→ thử tải local.")
    if not model_ok(MODEL_DIR):
        if shutil.disk_usage(MODEL_DIR).free < MODEL_SIZE + 512 * 2**20:
            raise OSError(
                f"Thiếu local disk cho snapshot AWQ; cần khoảng {MODEL_SIZE / 2**30:.2f} GiB ngoài thư viện."
            )
        print("Tải snapshot AWQ từ Hugging Face (không tải FP16).")
        download_script = (
            "from huggingface_hub import snapshot_download; "
            f"snapshot_download(repo_id={MODEL_REPO!r}, revision={MODEL_REVISION!r}, "
            f"local_dir={str(MODEL_DIR)!r}, allow_patterns={list(EXPECTED_FILES)!r}, max_workers=2)"
        )
        subprocess.run([str(PYTHON), "-c", download_script], check=True, timeout=3600)
        if not model_ok(MODEL_DIR):
            raise ValueError("Snapshot download thiếu files hoặc config không phải AWQ 4-bit.")
else:
    print("Dùng cache AWQ local.")
try:
    if not model_ok(DRIVE_DIR):
        print("Lưu AWQ lên Drive để tái sử dụng.")
        copy_model(MODEL_DIR, DRIVE_DIR)
except OSError as exc:
    print("Không lưu được weights lên Drive:", type(exc).__name__, "→ dùng local trong phiên này.")
# Cố định non-thinking ở server, kể cả client không gửi chat_template_kwargs.
template = json.loads((MODEL_DIR / "tokenizer_config.json").read_text())["chat_template"]
if not isinstance(template, str) or "enable_thinking" not in template:
    raise ValueError("Chat template không hỗ trợ enable_thinking; không start server.")
CHAT_TEMPLATE = ROOT / "nonthinking.jinja"
CHAT_TEMPLATE.write_text("{% set enable_thinking = false %}\n" + template, encoding="utf-8")
print("AWQ snapshot sẵn sàng:", MODEL_DIR)

## 3. Start API → nhận HTTPS URL

vLLM load AWQ8B từ local, float16 activations, context đã chọn (mặc định24K), một sequence,
engine V0/XFormers/eager. Giữ chunked prefill và prefix caching **tắt** để tránh nhánh
Triton prefix attention đã lỗi shared memory trên T4. Token budget bằng context vì
không chia chunk prefill; prompt dài có thể tăng peak VRAM. **Chưa xác nhận24K chạy được.**

Non-thinking cố định bằng official chat template của8B và request. Readiness10phút,
log lỗi, không retry vô hạn. Nếu thiếu KV-cache, báo capacity thực tế để chọn context thấp hơn.
Shared memory của kernel khác tổng VRAM, không dùng chung một hướng xử lý.

HTTPS chỉ lưu sau probe đúng alias và từ chối request thiếu key. `api_info.json`
tự cập nhật BASE_URL/API_KEY/MODEL_NAME; alias mới **qwen3-8b-awq-t4**.
Probe public chạy từ Colab, chưa xác minh từ máy DACNTT. Codex cần quyền/kết nối Drive
hoặc Drive sync để đọc file; mount Colab không tự chia sẻ filesystem với Codex.

[Nhánh backend](https://github.com/vllm-project/vllm/blob/v0.10.2/vllm/attention/backends/xformers.py#L565).


In [ ]:
# @title Start API
stop_api()
save_api_info("STARTING")
if not model_ok(MODEL_DIR) or not PYTHON.is_file():
    raise RuntimeError("Chạy bước 2 thành công trước.")
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError("Port 8000 đang được process khác dùng. Restart Session rồi chạy lại.")


def start_process(kind, command, env=None):
    with (ROOT / f"{kind}.log").open("w") as log:
        proc = subprocess.Popen(
            command, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True
        )
    (ROOT / f"{kind}.json").write_text(
        json.dumps({"pid": proc.pid, "starttime": process_identity(proc.pid), "binary": command[0]})
    )
    globals()[kind.upper() + "_PROCESS"] = proc
    return proc


def api_json(route, payload=None, base_url=None, timeout=180):
    url = (base_url or API + "/v1") + route
    response = requests.request(
        "GET" if payload is None else "POST",
        url,
        headers=HEADERS,
        json=payload,
        timeout=(10, timeout),
    )
    if not response.ok:
        detail = response.text.replace(API_KEY, "<REDACTED>")[:4000]
        raise RuntimeError(f"HTTP {response.status_code} {route}: {detail}")
    return response.json()


def server_error_details(log_text):
    lines = log_text.splitlines()
    # Lấy traceback/error trước block cảnh báo chat template và log shutdown.
    indices = [
        i
        for i, line in enumerate(lines)
        if re.search(
            r"\bERROR\b|Traceback \(most recent|(?:Runtime|Value|NotImplemented|Import)Error:|CUDA error|out of memory",
            line,
        )
    ]
    selected = set(range(max(0, len(lines) - 25), len(lines)))
    for i in indices[-4:]:
        selected.update(range(max(0, i - 3), min(len(lines), i + 35)))
    return "\n".join(lines[i] for i in sorted(selected)).replace(API_KEY, "<REDACTED>")[:18000]


def gpu_error_hint(error_text):
    # Chỉ xét thông báo lỗi, không quét INFO về dung lượng KV-cache để suy ra OOM.
    if "shared memory" in error_text.lower() and "outofresources" in error_text.lower():
        return (
            "Kernel vượt shared memory của GPU; đây không phải CUDA VRAM OOM. "
            "Bản T4 này đã tắt chunked prefill/prefix caching; nếu vẫn lỗi, gửi traceback gốc."
        )
    capacity = re.search(r"max seq len \((\d+)\).*KV cache \((\d+)\)", error_text, re.I)
    if capacity:
        return (
            f"Context yêu cầu {capacity[1]} vượt KV-cache {capacity[2]} tokens. "
            "Giảm CONTEXT_SIZE rồi start lại thủ công; không tăng VRAM utilization tự động."
        )
    if re.search(
        r"CUDA out of memory|torch\.OutOfMemoryError|No available memory for the cache blocks|"
        r"KV cache.*(?:too small|insufficient)|max seq len.*larger than.*KV cache",
        error_text,
        re.I,
    ):
        return "Lỗi VRAM/KV-cache: có thể chọn 8192 rồi thử lại thủ công."
    return ""


def chat_payload(prompt, **extra):
    return {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": 256,
        "stream": False,
        "chat_template_kwargs": {"enable_thinking": False},
        **extra,
    }


def final_content(data):
    message = data["choices"][0]["message"]
    content = message.get("content")
    if (
        not content
        or message.get("reasoning_content")
        or message.get("reasoning")
        or "<think>" in content
        or "</think>" in content
    ):
        raise ValueError("Thiếu final content hoặc model chưa tắt thinking.")
    if data["choices"][0].get("finish_reason") == "length":
        raise ValueError("Output bị cắt vì hết token budget.")
    return content


child_env = {k: v for k, v in os.environ.items() if not k.startswith("VLLM_") and k != "PYTHONPATH"}
child_env.update(
    {
        "VLLM_API_KEY": API_KEY,
        "VLLM_USE_V1": "0",
        "VLLM_ATTENTION_BACKEND": "XFORMERS",
        "VLLM_NO_USAGE_STATS": "1",
        "HF_HUB_OFFLINE": "1",
    }
)
command = [
    str(PYTHON),
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    str(MODEL_DIR),
    "--served-model-name",
    MODEL_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dtype",
    "half",
    "--quantization",
    "awq",
    "--max-model-len",
    str(CONTEXT_SIZE),
    "--max-num-seqs",
    "1",
    "--gpu-memory-utilization",
    "0.90",
    "--enforce-eager",
    "--no-enable-chunked-prefill",
    "--no-enable-prefix-caching",
    "--max-num-batched-tokens",
    str(CONTEXT_SIZE),
    "--generation-config",
    "vllm",
    "--chat-template",
    str(CHAT_TEMPLATE),
    "--disable-log-requests",
]
try:
    SERVER_PROCESS = start_process("server", command, child_env)
    deadline = time.monotonic() + 600
    for _ in range(300):
        if SERVER_PROCESS.poll() is not None:
            raise RuntimeError("vLLM server đã thoát; xem log bên dưới.")
        try:
            if requests.get(API + "/health", timeout=2).status_code == 200:
                break
        except requests.RequestException:
            pass
        if time.monotonic() >= deadline:
            raise TimeoutError("Model chưa sẵn sàng sau 600 giây.")
        time.sleep(2)
    else:
        raise TimeoutError("Model readiness timeout.")
    models = api_json("/models")
    found = next((m for m in models.get("data", []) if m["id"] == MODEL_NAME), None)
    if found is None or found.get("max_model_len") != CONTEXT_SIZE:
        raise ValueError("API model/context khác cấu hình yêu cầu.")
    print(
        "Model trả lời:",
        final_content(
            api_json("/chat/completions", chat_payload("Chào bạn! Trả lời ngắn bằng tiếng Việt."))
        ),
    )
    SERVER_READY = True
    print("LOCAL READY:", API + "/v1", "| Context:", CONTEXT_SIZE, "| GPU:", GPU["name"])
except BaseException as error:
    # Chụp lỗi trước khi cleanup tạo thêm log SIGTERM/NCCL/Ray không phải nguyên nhân gốc.
    log_path = ROOT / "server.log"
    error_log = log_path.read_text(errors="replace") if log_path.exists() else ""
    print("=== vLLM error log trước khi dừng server ===")
    print(server_error_details(error_log))
    try:
        stop_api()
        save_api_info("FAILED")
    except Exception as cleanup_error:
        print("Cleanup:", str(cleanup_error).replace(API_KEY, "<REDACTED>")[:500])
    # Lỗi engine khởi động có thể chỉ trả "server đã thoát"; lấy cả ERROR/exception gốc.
    error_lines = "\n".join(
        line
        for line in error_log.splitlines()
        if re.search(r"\bERROR\b|(?:Value|Runtime|OutOfMemory)Error:", line)
    )
    hint = gpu_error_hint(str(error) + "\n" + error_lines)
    if hint:
        print(hint)
    print("Gửi response lỗi HTTP và đoạn ERROR/Traceback ở trên; log đầy đủ:", log_path)
    raise

BASE_URL = None
if ENABLE_HTTPS:
    try:
        cloudflared = ROOT / "cloudflared"
        expected_hash = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"
        # Chỉ hash binary tunnel ~40 MB; không hash weights ~9 GB.
        if (
            not cloudflared.exists()
            or hashlib.sha256(cloudflared.read_bytes()).hexdigest() != expected_hash
        ):
            url = "https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64"
            response = requests.get(url, timeout=(15, 120))
            if (
                response.status_code != 200
                or hashlib.sha256(response.content).hexdigest() != expected_hash
            ):
                raise RuntimeError("Không tải/xác minh được cloudflared đã pin.")
            cloudflared.write_bytes(response.content)
            cloudflared.chmod(0o755)
        TUNNEL_PROCESS = start_process(
            "tunnel",
            [str(cloudflared), "tunnel", "--url", API, "--no-autoupdate", "--protocol", "http2"],
        )
        deadline = time.monotonic() + 90
        for _ in range(45):
            if TUNNEL_PROCESS.poll() is not None:
                raise RuntimeError("Cloudflare Tunnel đã thoát.")
            log_text = (ROOT / "tunnel.log").read_text(errors="replace")
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
            if match:
                candidate = match[0] + "/v1"
                try:
                    models = api_json("/models", base_url=candidate, timeout=5)
                    denied = requests.get(candidate + "/models", timeout=(5, 5))
                    if MODEL_NAME in [
                        m["id"] for m in models.get("data", [])
                    ] and denied.status_code in (401, 403):
                        BASE_URL = candidate
                        break
                except (requests.RequestException, ValueError, RuntimeError):
                    pass
            if time.monotonic() >= deadline:
                break
            time.sleep(2)
        if BASE_URL is None:
            raise TimeoutError("Chưa xác minh được HTTPS API trong 90 giây.")
    except Exception as exc:
        stop_process("tunnel")
        print(
            "HTTPS chưa khả dụng:",
            str(exc).replace(API_KEY, "<REDACTED>")[:500],
            "— chỉ dùng local; xem tunnel.log.",
        )
try:
    save_api_info("READY" if BASE_URL else "LOCAL_ONLY", BASE_URL)
except OSError:
    stop_api()
    raise RuntimeError("Không lưu được api_info.json trên Drive; đã dừng API.") from None
print("BASE_URL:", BASE_URL or "UNAVAILABLE — local API only")
print("Đã lưu BASE_URL, API_KEY (ẩn), MODEL_NAME tại:", INFO_PATH)
print("URL chỉ có hiệu lực khi runtime/server/tunnel còn chạy.")

## 4. Test API

Hai request tuần tự: tiếng Việt và JSON số liệu mẫu, dùng HTTPS nếu có hoặc báo rõ local.
Hiển thị tokens, latency, **output tokens/giây end-to-end** (gồm cả xử lý prompt/network),
không gọi đó là tốc độ decode riêng của model. VRAM là snapshot sau request, không phải peak.
UNKNOWN khi thiếu số đo. Đây là engineering smoke, không phải benchmark tài chính/M8.


In [ ]:
# @title Test API
if not SERVER_READY:
    raise RuntimeError("Start API thành công trước khi test.")
TEST_BASE = BASE_URL or API + "/v1"
print("Kiểm tra qua:", TEST_BASE)


def check_answer(content):
    if "hà nội" not in content.casefold():
        raise ValueError("Chưa nhận được đáp án Hà Nội.")


def check_json(content):
    value = json.loads(content)
    expected = {"metric": "gross_margin", "numerator": 100, "denominator": 1000, "percentage": 10}
    if not isinstance(value, dict) or not expected.keys() <= value.keys():
        raise ValueError("Thiếu JSON keys.")
    if value["metric"] != "gross_margin" or any(
        type(value[k]) not in (int, float) or value[k] != expected[k]
        for k in ("numerator", "denominator", "percentage")
    ):
        raise ValueError("JSON sai kiểu số hoặc giá trị.")


tests = [
    ("Tiếng Việt", chat_payload("Thủ đô Việt Nam là thành phố nào? Trả lời ngắn."), check_answer),
    (
        "JSON",
        chat_payload(
            'Chỉ trả JSON: metric="gross_margin", numerator=100, denominator=1000, percentage=10.',
            response_format={"type": "json_object"},
        ),
        check_json,
    ),
]
rows = []
for name, payload, checker in tests:
    row = {
        "Test": name,
        "Status": "FAIL",
        "Input tokens": "UNKNOWN",
        "Output tokens": "UNKNOWN",
        "Output tok/s (E2E)": "UNKNOWN",
        "GPU used MiB": "UNKNOWN",
        "GPU free MiB": "UNKNOWN",
        "Error": "",
    }
    started = time.perf_counter()
    try:
        data = api_json("/chat/completions", payload, base_url=TEST_BASE, timeout=180)
        usage = data.get("usage") or {}
        row["Input tokens"] = usage.get("prompt_tokens", "UNKNOWN")
        row["Output tokens"] = usage.get("completion_tokens", "UNKNOWN")
        elapsed = time.perf_counter() - started
        if isinstance(row["Output tokens"], int) and elapsed > 0:
            row["Output tok/s (E2E)"] = round(row["Output tokens"] / elapsed, 2)
        content = final_content(data)
        print(name, "→", content[:1000])
        checker(content)
        row["Status"] = "PASS"
    except requests.Timeout:
        stop_api()
        row["Error"] = "Request timeout: đã dừng API để tránh inference tiếp tục ngầm."
    except Exception as exc:
        row["Error"] = str(exc).replace(API_KEY, "<REDACTED>")[:500]
    row["Latency s"] = round(time.perf_counter() - started, 3)
    memory = gpu_memory()
    row["GPU used MiB"], row["GPU free MiB"] = memory["used_MiB"], memory["free_MiB"]
    rows.append(row)
    if not SERVER_READY:
        break
display(pd.DataFrame(rows).fillna("UNKNOWN"))

### Kết thúc phiên

**Run all không dừng API**. Khi dùng xong, bật ô Stop rồi chạy riêng cell dưới.
Restart Session: Runtime → Disconnect and delete runtime; sau đó chạy từ đầu.
Khi phiên bị ngắt đột ngột, `api_info.json` có thể còn URL cũ: client cần kiểm tra
`/v1/models` trước khi dùng; chạy lại Start API sẽ tự ghi thông tin mới.
Nếu lỗi, gửi traceback/đoạn cuối log đã che secrets, **không gửi `api_info.json`**.

In [ ]:
# fmt: off
#@title Stop API (chỉ chạy khi kết thúc)
STOP_API = False #@param {type:"boolean"}
# fmt: on
if STOP_API:
    stop_api()
    print("Đã dừng server/tunnel và đánh dấu STOPPED trong api_info.json.")